<a href="https://colab.research.google.com/github/Chirag-GH/afcat-result-calculator/blob/main/afcat_result_calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Click on the Play Button ▶︎, and Upload your Answer Key.
!pip install -q PyMuPDF

import re
import fitz
import pandas as pd
from google.colab import files

GREEN = 4245067

uploaded_file = files.upload()
PATH = list(uploaded_file.keys())[0]

doc = fitz.open(PATH)

rows = []
current_row = {}
correct_ans = None

for page in doc:
    data = page.get_text("dict")

    for block in data["blocks"]:
        if "lines" not in block:
            continue

        for line in block["lines"]:
            spans = line["spans"]
            line_text = "".join(span["text"] for span in spans).strip()

            # New question
            q_match = re.match(r"Q\.(\d+)", line_text)

            if q_match:
                current_row = {
                    "question_id": None,
                    "correct_ans": None,
                    "chosen_ans": None
                }
                correct_ans = None

            # Options
            option_match = re.match(r"([1-4])\.\s*(.*)", line_text)

            # Correct answer
            if option_match and any(span["color"] == GREEN for span in spans):
                correct_ans = option_match.group(1)

            # Question ID
            if "Question ID" in line_text:
                numbers = re.findall(r"\d+", line_text)

                if numbers:
                    current_row["question_id"] = numbers[-1]
                    current_row["correct_ans"] = correct_ans

            # Chosen answer
            if "Chosen Option" in line_text:
                chosen_match = re.findall(r"[1-4]|--", line_text)

                if chosen_match:
                    current_row["chosen_ans"] = chosen_match[0]
                    rows.append(current_row)

df = pd.DataFrame(rows)

correct_filt = df["correct_ans"] == df["chosen_ans"]
unaswered_filt = df["chosen_ans"] == "--"
incorrect_filt = ~unaswered_filt & ~correct_filt

correct_count = len(df[correct_filt])
unanswered_count = len(df[unaswered_filt])
incorrect_count = len(df[incorrect_filt])

attempt_count = correct_count + incorrect_count

final_score = (correct_count * 3) - incorrect_count #+3 for correct, -1 for incorrect

print(f"""
{"Attempted":12}: {attempt_count}
{"Correct":12}: {correct_count}
{"Incorrect":12}: {incorrect_count}
{"Unanswered":12}: {unanswered_count}
""")

print(f"Final Score : {final_score}")